# Spotify Playlist EDA

In [ ]:
cd '/Users/wiseer/Documents/github/listen-wiseer/src/'

In [ ]:
from analysis.data import *
from analysis.genre import *
from analysis.plotting import *

from utils.const import *

data = LoadPlaylistData()

In [ ]:
df = data.return_enoa_data()
new_genres_list = set(df[df.gen_4.isnull()]["first_genre"])
playlist_first_genre_map, enoa_sub_genre_map = data.create_subgenre_maps()

In [ ]:
# parameters for zouk train
df = df[df.playlist_name.isin(["zoukini", "zoukini flow"])].drop_duplicates('track_name')
df.loc[df.playlist_name == "zoukini flow", "flow"] = 1
df.flow.fillna(0, inplace=True)
df = df[df.time_signature == 4]
df = df[df.danceability >= 0.5]
df = df[((df.tempo > 100) & (df.tempo < 175))]
df = df[((df.energy >= 0.3) & (df.energy < 0.7))]

In [ ]:
# plot my genres
genre_groups = (df.groupby("first_genre")[["top", "left"]].agg({"min", "max"}))
genre_groups.columns = genre_groups.columns.map("_".join)
plot_new_genres(genre_groups, df, new_genres_list, "first_genre")

In [ ]:
df[df.energy<0.3]

In [ ]:
df.energy.hist()

In [ ]:
df[df.danceability < 0.5]

In [ ]:
df.tempo.hist()

In [ ]:
df[df.track_name == "Float"]["time_signature"]

In [ ]:
df[df.track_name == "Float"]["tempo"]

In [ ]:
df[df.track_name=="My Oasis (feat. Burna Boy)"]['tempo']

# float sol rising
# my oasis burna boy

## 1 | Plot ENOA data

In [ ]:
# group genres to play min, max coordinates
genre_groups = df.groupby("gen_4")[["top", "left"]].agg({"min", "max"})
genre_groups.columns = genre_groups.columns.map("_".join)
plot_enoa_area(genre_groups)
plt.savefig("analysis/output/genres/enoa_gen_4.png")

In [ ]:
# group genres to play min, max coordinates
genre_groups = df.groupby("gen_8")[["top", "left"]].agg({"min", "max"})
genre_groups.columns = genre_groups.columns.map("_".join)
plot_enoa_area(genre_groups)
plt.savefig("analysis/output/genres/enoa_gen_8.png")

In [ ]:
# group genres to play min, max coordinates
genre_groups = df.groupby("my_genre")[["top", "left"]].agg({"min", "max"})
genre_groups.columns = genre_groups.columns.map("_".join)
plot_enoa_area(genre_groups)
plt.savefig("analysis/output/genres/enoa_my_genre.png")

In [ ]:
# plot new genres
plot_enao_new_genres(
    df, new_genres_list, group="gen_8"
)  # this looks like my_genres not 8
plt.savefig("analysis/output/genres/enoa_new_genres.png")

## 2 | Update genre map

In [ ]:
dm = calculate_first_genre_distances(df, ["top", "left"])
sns.heatmap(dm)

In [ ]:
# return new genre df to review
new_genres_df = return_new_genres_df(df)

# NOTE: this is just an old trick to compare and should be removed
old_map = pd.read_csv(
    "/Users/wiseer/Documents/github/listen-wiseer/src/data/genres/genre_map_old.csv",
    index_col=0,
)
new_genres_df = new_genres_df.merge(old_map, on="first_genre", how="left")

# append best matches
new_genres_df = append_best_playlist_match(new_genres_df, playlist_first_genre_map, dm)
# NOTE: this is based on enoa distance!!!
new_genres_df = append_best_enoa_match(
    new_genres_df, playlist_first_genre_map, enoa_sub_genre_map, dm
)
new_genres_df = new_genres_df.drop_duplicates(["artist_names", "first_genre"])
new_genres_df.to_csv(
    "/Users/wiseer/Documents/github/listen-wiseer/src/analysis/output/genres/genre_map_review.csv"
)  # TODO: add date?
new_genres_df[:30]

# TODO: how to select which is the best match to automate update? At least see if there is overlap between the two

In [ ]:
plot_playlist_genres(df, "cooking with palms trax")

In [ ]:
# plot my genres
genre_groups = (
    df[df.gen_8 == "rock"]
    .groupby("first_genre")[["top", "left"]]
    .agg({"min", "max"})
)
genre_groups.columns = genre_groups.columns.map("_".join)
plot_new_genres(genre_groups, df, new_genres_list, "first_genre")

In [ ]:
# group on the level you want to plot new genres
genre_groups = (
    df[df.gen_4 == "electronic"]
    .groupby("sub_genre")[["top", "left"]]
    .agg({"min", "max"})
)
genre_groups.columns = genre_groups.columns.map("_".join)
genre_groups.reset_index(inplace=True)
plot_new_genres(genre_groups, df, new_genres_list,'sub_genre')

## 3 | Review feature distribution by genre

In [ ]:
plot_pairplot(df, hue="gen_4")
plt.savefig("analysis/output/genres/gen_4_pairplot.png")

In [ ]:
plot_pairplot(df, hue="gen_8")
plt.savefig("analysis/output/genres/gen_8_pairplot.png")

## 4 | Analyze Playlists

In [ ]:
boxplot_playlist_by_decade(df)
plt.savefig("analysis/output/playlists/playlist_decade_popularity.png")

In [ ]:
plot_playlist_artist_popularity(df)
plt.savefig("analysis/output/playlists/playlist_artist_popularity.png")

In [ ]:
# TODO: check if this is the best way to group playlists
df.groupby("playlist_name").gen_4.value_counts().unstack().plot(
    kind="barh", stacked=True
)
plt.savefig("analysis/output/playlists/playlist_gen_4.png")

In [ ]:
df.groupby("playlist_name").gen_8.value_counts().unstack().plot(
    kind="barh", stacked=True
)
plt.savefig("analysis/output/playlists/playlist_gen_8.png")

In [ ]:
# let's redo this to one graph stacked
plot_my_genre_by_playlist_group(df, playlist_group_dict, order=None)
plt.savefig("analysis/output/playlists/playlist_genres.png")

In [ ]:
df["playlist_group"] = df["playlist_name"].map(
    lambda x: next((k for k, v in playlist_group_dict.items() if x in v), None)
)
plot_pairplot(df, hue="playlist_group")
plt.savefig("analysis/output/playlists/playlist_pairplot.png")

## 5 | Outlier Analysis

In [ ]:
outliers = data.return_outlier_df(df)
outliers.to_csv("/Users/wiseer/Documents/github/listen-wiseer/src/analysis/output/outliers/outlier_df.csv")

In [ ]:
plt.figure(figsize=(14, 6))
sns.histplot(outliers, x="score", hue="playlist_name", kde=True, multiple="stack")
plt.savefig("analysis/output/outliers/outlier_hist.png")

In [ ]:
plot_outlier_hist_subplots(outliers)
plt.savefig("analysis/output/outliers/outlier_hist.png")

In [ ]:
plot_outlier_enoa(outliers)
plt.savefig("analysis/output/outliers/outlier_enoa_scatterplot.png")

## 6 | Dimensionality Reduction

In [ ]:
from sklearn.manifold import TSNE


def calculate_tsne(df, playlist):
    data = df[df["playlist_name"] == playlist].copy()
    data.dropna(subset=num_features, inplace=True)
    tsne = TSNE(n_components=2, random_state=0)
    X_tsne = tsne.fit_transform(data[num_features])

    # group
    data["X_tsne"] = X_tsne[:, 0]
    data["y_tsne"] = X_tsne[:, 1]

    # assign groups
    data.loc[data["X_tsne"] < 0, "tsne"] = 1
    data.loc[data["X_tsne"] >= 0, "tsne"] = 2

    return data


def plot_tsne(data, axes):
    sns.scatterplot(
        x="X_tsne", y="y_tsne", hue="tsne", data=data, palette="viridis", ax=axes[0]
    )


def plot_tsne_by_genres(data, var, axes):
    plt.figure(figsize=(10, 8))
    if any(data[var] < 0):
        df_counts = (
            data[data[var] < 0]
            .groupby(["first_genre"])
            .size()
            .reset_index(name="Count")
        )
        sns.barplot(
            x="Count",
            y="first_genre",
            data=df_counts,
            orient="h",
            color="purple",
            ax=axes[1],
        )
    if any(data[var] > 0):
        df_counts = (
            data[data[var] >= 0]
            .groupby(["first_genre"])
            .size()
            .reset_index(name="Count")
        )
        sns.barplot(
            x="Count",
            y="first_genre",
            data=df_counts,
            orient="h",
            color="orange",
            ax=axes[1],
        )

In [ ]:
    playlist='zoukini'
    tsne_data = calculate_tsne(df, playlist)
    # Plot tsne on the left column
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))
    plot_tsne(tsne_data, axes)
    axes[0].set_title(f"TSNE Plot for {playlist}")

    # Plot tsne_by_genre on the right column
    plot_tsne_by_genres(tsne_data, 'X_tsne', axes)
    axes[1].set_title(f"TSNE by Genre for {playlist}")

    # Adjust layout
    plt.tight_layout()
    plt.show()

## Save analysis to pdf

In [ ]:
%%bash
jupyter nbconvert --to html --no-input analysis/output/eda.ipynb
echo "Conversion completed."

In [ ]:
from datetime import datetime

import pdfkit

# Convert HTML to PDF
current_date_str = datetime.now().strftime("%Y-%m-%d")
html_file_path = "analysis/output/eda.html"
pdf_output_path = f"analysis/output/eda_{current_date_str}.pdf"
pdfkit.from_file(html_file_path, pdf_output_path)
print(f"PDF file has been created: {pdf_output_path}")